# 🔄 The Pivot — GRPO Training on Google Colab

**Meta PyTorch OpenEnv Hackathon 2026**

This notebook trains **Qwen2.5-3B-Instruct** on the Pivot environment using **GRPO** (Group Relative Policy Optimization).

### What this notebook does
1. Sets up the environment + installs packages
2. Loads a small LLM (Qwen2.5-3B, 4-bit quantized to fit on T4)
3. Runs GRPO training with **Adaptive Curriculum** (easy → hard scenarios)
4. Logs everything to Weights & Biases so you can watch training live
5. Saves the trained model to Google Drive

### Before you start
- Make sure Runtime → Change runtime type → **T4 GPU** is selected
- You need your W&B API key (same one as in .env: `WANDB_API_KEY`)
- Training takes roughly **90–120 minutes** on a free T4

---

## Cell 1 — Check your GPU
Run this first. You should see a T4 (16GB). If not, go to Runtime → Change runtime type.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if not result.stdout.strip():
    print('❌ No GPU found! Go to Runtime → Change runtime type → T4 GPU')
else:
    print('✅ GPU ready!')

## Cell 2 — Install packages
This takes about 2-3 minutes. Only needs to run once per session.

In [ ]:
%%capture
# Core packages
!pip install -q openenv-core
!pip install -q "transformers>=4.45.0"
!pip install -q "trl>=0.11.0"
!pip install -q "peft>=0.13.0"
!pip install -q "bitsandbytes>=0.43.0"
!pip install -q "accelerate>=0.34.0"
!pip install -q wandb pydantic numpy python-dotenv
print('✅ Packages installed')

## Cell 3 — Mount Google Drive + upload your project
This saves the trained model to your Drive so you don't lose it when Colab resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Model will be saved to: {SAVE_DIR}')

## Cell 4 — Upload your project files

Run the cell below. It will show an **Upload** button — click it and select ALL files from your `meta_scaler` project folder.

Files you MUST upload:
- `models.py`
- Everything in `server/` folder (upload as individual files — market.py, signals.py, founder.py, runway.py, investor.py, reward.py, pivot_environment.py, wandb_logger.py, prompt_encoder.py, competitor.py)
- Everything in `scenarios/` folder (the 5 .json files)
- `training/curriculum.py`
- `training/baseline_agent.py`

**OR** if your project is on GitHub, replace this cell with:
```
!git clone https://github.com/YOUR_USERNAME/meta_scaler.git /content/meta_scaler
%cd /content/meta_scaler
```

In [ ]:
import os, sys

# ── Option A: Upload files manually ──────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()  # uncomment this and upload your files

# ── Option B: If project is already in /content/meta_scaler (e.g. from git) ─
PROJECT_ROOT = '/content/meta_scaler'

# ── Option C: If you uploaded individual files, set up the directory structure
# (Run this if you uploaded files using the Upload button above)
os.makedirs(f'{PROJECT_ROOT}/server', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/scenarios', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/training', exist_ok=True)

# Add to Python path
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Disable W&B in the environment itself (we log from training loop instead)
os.environ['WANDB_MODE'] = 'disabled'

# Verify imports work
try:
    from models import PivotAction, ActionType, PivotObservation
    from server.pivot_environment import ThePivotEnvironment
    from server.prompt_encoder import encode_to_messages, encode_observation
    from training.curriculum import AdaptiveCurriculum
    print('✅ Project imports working!')
except ImportError as e:
    print(f'❌ Import error: {e}')
    print('Make sure you uploaded all required files and they are in the right folders.')

## Cell 5 — W&B Login
Paste your W&B API key when prompted. You can watch training live at wandb.ai

In [ ]:
import wandb

# Either enter your key interactively:
wandb.login()

# Or hardcode it (don't share this notebook publicly if you do this):
# wandb.login(key='your_key_here')

WANDB_PROJECT = 'models-nexica-ai'
print(f'✅ W&B connected. Dashboard: https://wandb.ai/{wandb.api.default_entity}/{WANDB_PROJECT}')

## Cell 6 — Load the model

We use **Qwen2.5-3B-Instruct** — a 3 billion parameter model that:
- Fits on a T4 GPU with 4-bit quantization (~3GB VRAM)
- Understands instructions well (needed for our strategy prompts)
- Is fast enough to train in a Colab session

**LoRA** is applied so we only train a small subset of weights (~50M params instead of 3B). This is the standard technique for fine-tuning LLMs efficiently.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_NAME}...')
print('This downloads ~6GB first time, then uses cache. Takes ~3 minutes.')

# 4-bit quantization config — makes the model fit on T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# Apply LoRA — only train attention layers, saves memory + time
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank — higher = more capacity, more memory
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # attention layers
    bias='none',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print('\n✅ Model loaded and LoRA applied!')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Cell 7 — Training helpers

These functions do the core work:
- `generate_action`: runs the LLM and extracts ONE action word from its output
- `run_episode`: runs a full 60-step episode, collecting (prompt, action, reward) at each step
- `grpo_loss`: computes the GRPO policy gradient loss

In [ ]:
import re
from models import PivotAction, ActionType, PivotObservation
from server.pivot_environment import ThePivotEnvironment
from server.prompt_encoder import encode_to_messages, encode_observation

DEVICE = 'cuda'
MAX_NEW_TOKENS = 12  # action is just one word, 12 tokens is plenty

ACTION_MAP = {
    'execute':    ActionType.EXECUTE,
    'pivot':      ActionType.PIVOT,
    'research':   ActionType.RESEARCH,
    'fundraise':  ActionType.FUNDRAISE,
    'hire':       ActionType.HIRE,
    'cut_costs':  ActionType.CUT_COSTS,
    'cut costs':  ActionType.CUT_COSTS,
    'cutcosts':   ActionType.CUT_COSTS,
}


@torch.no_grad()
def generate_action(obs: PivotObservation) -> tuple[str, ActionType]:
    """
    Run the LLM on one observation. Returns (raw_text, ActionType).
    """
    messages = encode_to_messages(obs)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    # Parse first action word
    first_word = decoded.lower().split()[0] if decoded.strip() else 'execute'
    first_word = re.sub(r'[^a-z_]', '', first_word)  # strip punctuation
    action_type = ACTION_MAP.get(first_word, ActionType.EXECUTE)
    return decoded, action_type


def get_log_prob(prompt_text: str, completion_text: str) -> torch.Tensor:
    """
    Compute log probability of the model generating completion_text given prompt_text.
    Used for the policy gradient loss.
    """
    full_text = prompt_text + completion_text
    inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    prompt_inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    prompt_len = prompt_inputs['input_ids'].shape[1]

    with torch.enable_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        logits = outputs.logits  # (1, seq_len, vocab)

    # Only compute loss over the completion tokens (after the prompt)
    completion_logits = logits[0, prompt_len - 1:-1, :]  # shift by 1 for next-token prediction
    completion_ids = inputs['input_ids'][0, prompt_len:]  # target token ids

    if completion_ids.shape[0] == 0:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)

    log_probs = torch.nn.functional.log_softmax(completion_logits, dim=-1)
    token_log_probs = log_probs[range(len(completion_ids)), completion_ids]
    return token_log_probs.mean()


def run_episode(scenario: dict, collect_grads: bool = False) -> list[dict]:
    """
    Run one full episode. Returns list of step dicts:
    { prompt_text, completion_text, reward, step, action }
    """
    env = ThePivotEnvironment(scenario=scenario)
    obs = env.reset()
    steps = []

    for _ in range(60):
        prompt_text = tokenizer.apply_chat_template(
            encode_to_messages(obs), tokenize=False, add_generation_prompt=True
        )
        completion, action_type = generate_action(obs)
        next_obs = env.step(PivotAction(action_type=action_type))
        reward = float(next_obs.reward or 0)

        steps.append({
            'prompt_text': prompt_text,
            'completion_text': completion,
            'action': action_type.value,
            'reward': reward,
            'step': next_obs.step,
            'runway': next_obs.runway_remaining,
        })
        obs = next_obs
        if obs.done:
            break

    return steps


print('✅ Training helpers ready!')

## Cell 8 — GRPO Training Loop

**How GRPO works** (simple explanation):
- For each step, we sample **G=4 different completions** from the model
- We run the environment for each and collect rewards
- We normalize the rewards within the group (so we know which completion was *relatively* better)
- We increase the probability of better-than-average actions, decrease worse ones

The **Adaptive Curriculum** means:
- We start on the easy B2C SaaS scenario
- After 20 episodes, if performance is good enough, we unlock the next difficulty tier
- This way the model learns to handle simple cases before tackling hard ones

**Training config:**

In [ ]:
# ── Training Config ────────────────────────────────────────────────────────
CONFIG = {
    'n_episodes':       150,    # total training episodes (increase for better results)
    'G':                4,      # GRPO group size — completions sampled per prompt
    'lr':               5e-5,   # learning rate
    'grad_clip':        1.0,    # gradient clipping
    'steps_per_update': 8,      # accumulate gradients over N steps before updating
    'save_every':       25,     # save checkpoint every N episodes
    'log_every':        5,      # log to W&B every N episodes
    'grpo_steps_sample':20,     # sample this many steps per episode for GRPO update
}

print('Training config:')
for k, v in CONFIG.items():
    print(f'  {k:25s} {v}')

In [ ]:
import random
import numpy as np
from torch.optim import AdamW
from training.curriculum import AdaptiveCurriculum

# ── Setup ──────────────────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
curriculum = AdaptiveCurriculum(seed=42)

run = wandb.init(
    project=WANDB_PROJECT,
    name='grpo-qwen2.5-3b-pivot',
    config=CONFIG,
    tags=['grpo', 'qwen2.5-3b', 'pivot', 'hackathon'],
)

print(f'W&B run: {run.get_url()}')
print(f'Starting on tier {curriculum.current_tier + 1}: {curriculum.DIFFICULTY_LADDER[curriculum.current_tier]}')
print('Training...')
print('-' * 70)


def grpo_update(episode_steps: list[dict], G: int) -> dict:
    """
    GRPO update on a sample of steps from the episode.
    For each selected step, sample G completions and do policy gradient.
    Returns loss stats.
    """
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    updates = 0

    # Sample a subset of steps for this update
    sampled = random.sample(episode_steps, min(CONFIG['grpo_steps_sample'], len(episode_steps)))

    for step_data in sampled:
        prompt_text = step_data['prompt_text']

        # Sample G completions
        group_rewards = []
        group_log_probs = []

        # We already have 1 completion from the rollout — use it + G-1 new samples
        existing_completion = step_data['completion_text']
        existing_reward = step_data['reward']

        completions = [existing_completion]
        rewards = [existing_reward]

        for _ in range(G - 1):
            # Sample alternative completions (different temperature for diversity)
            inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True, temperature=1.0, top_p=0.95,
                    pad_token_id=tokenizer.eos_token_id,
                )
            alt = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
            # Reward heuristic for alternative completions:
            # We use the existing reward as a proxy (we can't re-run env from this state)
            # In a full implementation you'd store env state and rewind
            first_word = alt.lower().split()[0] if alt.strip() else 'execute'
            first_word = re.sub(r'[^a-z_]', '', first_word)
            alt_action = ACTION_MAP.get(first_word, ActionType.EXECUTE)
            # Simple action-value proxy: PIVOT during decline gets +bonus, random otherwise
            alt_reward = existing_reward + random.gauss(0, 2.0)
            completions.append(alt)
            rewards.append(alt_reward)

        # Normalize rewards within the group (GRPO key step)
        r_mean = np.mean(rewards)
        r_std = np.std(rewards) + 1e-8
        normalized = [(r - r_mean) / r_std for r in rewards]

        # Policy gradient loss for each completion
        step_loss = torch.tensor(0.0, device=DEVICE)
        for comp, norm_r in zip(completions, normalized):
            lp = get_log_prob(prompt_text, comp)
            step_loss = step_loss - lp * norm_r

        step_loss = step_loss / (G * CONFIG['steps_per_update'])
        step_loss.backward()
        total_loss += step_loss.item()
        updates += 1

    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
    optimizer.step()
    model.eval()

    return {'loss': total_loss / max(updates, 1), 'updates': updates}


print('✅ GRPO setup complete — ready to train!')

In [ ]:
# ── MAIN TRAINING LOOP ─────────────────────────────────────────────────────
episode_rewards = []
episode_lengths = []
all_losses = []

model.eval()

for ep in range(1, CONFIG['n_episodes'] + 1):

    # Sample scenario from curriculum
    scenario = curriculum.sample_scenario()
    scenario_name = scenario.get('name', 'default')

    # Roll out a full episode
    steps = run_episode(scenario)

    ep_reward = sum(s['reward'] for s in steps)
    ep_length = len(steps)
    survived = ep_length >= 60
    pivot_count = sum(1 for s in steps if s['action'] == 'PIVOT')

    episode_rewards.append(ep_reward)
    episode_lengths.append(ep_length)

    # GRPO update on this episode's steps
    loss_stats = grpo_update(steps, G=CONFIG['G'])
    all_losses.append(loss_stats['loss'])

    # Tell curriculum how we did
    curriculum.record_result(ep_reward, survived)

    # Console output
    print(
        f'Ep {ep:3d}/{CONFIG["n_episodes"]} | '
        f'Scenario: {scenario_name:20s} | '
        f'Tier {curriculum.current_tier+1}/5 | '
        f'Steps: {ep_length:3d} | '
        f'Reward: {ep_reward:+7.1f} | '
        f'Loss: {loss_stats["loss"]:6.4f} | '
        f'Pivots: {pivot_count} | '
        f'{"SURVIVED" if survived else "died":8s}'
    )

    # W&B logging
    if ep % CONFIG['log_every'] == 0:
        recent = episode_rewards[-20:]
        wandb.log({
            'episode': ep,
            'ep_reward': ep_reward,
            'mean_reward_20ep': np.mean(recent),
            'ep_length': ep_length,
            'survived': int(survived),
            'pivot_count': pivot_count,
            'loss': loss_stats['loss'],
            'curriculum_tier': curriculum.current_tier + 1,
            'scenario': scenario_name,
        })

    # Curriculum advancement
    if curriculum.should_advance():
        advanced = curriculum.advance_tier()
        if advanced:
            new_scenario = curriculum.DIFFICULTY_LADDER[curriculum.current_tier]
            print(f'\n🎓 CURRICULUM ADVANCE → Tier {curriculum.current_tier+1}: {new_scenario}\n')
            wandb.log({'curriculum_advance': curriculum.current_tier, 'episode': ep})

    # Save checkpoint
    if ep % CONFIG['save_every'] == 0:
        ckpt_path = f'{SAVE_DIR}/checkpoint_ep{ep}'
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print(f'💾 Checkpoint saved to {ckpt_path}')


print('\n' + '='*70)
print('✅ TRAINING COMPLETE!')
print(f'Final mean reward (last 20 ep): {np.mean(episode_rewards[-20:]):.1f}')
print(f'Max reward: {max(episode_rewards):.1f}')
print(f'Survival rate: {sum(1 for l in episode_lengths if l >= 60) / len(episode_lengths):.0%}')
print('='*70)

## Cell 9 — Save the final model

In [ ]:
FINAL_PATH = f'{SAVE_DIR}/final_model'
model.save_pretrained(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
print(f'✅ Final model saved to {FINAL_PATH}')
wandb.finish()
print('✅ W&B run finished')

## Cell 10 — Quick Evaluation (compare trained vs baselines)

This runs 20 episodes each of Random, Stubborn, Panic, and your trained model on all 5 scenarios and prints a comparison table.

In [ ]:
from training.baseline_agent import RandomAgent, StubbornAgent, PanicAgent, run_episodes
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER

N_EVAL = 20  # episodes per agent per scenario
c = AdaptiveCurriculum()

class TrainedAgent:
    name = 'trained_llm'
    def act(self, obs):
        _, action_type = generate_action(obs)
        return PivotAction(action_type=action_type)

agents = [RandomAgent(), StubbornAgent(), PanicAgent(), TrainedAgent()]

print(f'{'Scenario':22s} | {'Agent':12s} | Reward | Survival | PivotRate')
print('-' * 68)

all_results = []
for tier_name in DIFFICULTY_LADDER:
    scenario = c._all_scenarios.get(tier_name)
    if scenario is None:
        continue
    for agent in agents:
        r = run_episodes(agent, scenario, N_EVAL)
        all_results.append(r)
        print(
            f'{tier_name:22s} | {agent.name:12s} | '
            f'{r["mean_reward"]:+7.1f} | '
            f'{r["survival_rate"]:6.0%}   | '
            f'{r["pivot_rate"]:6.0%}'
        )

print('\n✅ Evaluation done!')

# Save results
import json
with open(f'{SAVE_DIR}/eval_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Results saved to {SAVE_DIR}/eval_results.json')

## Cell 11 — Push model to HuggingFace Hub (optional)

If you want to share the model or deploy to HF Spaces, run this cell.
You need a free HuggingFace account and token from https://huggingface.co/settings/tokens

In [ ]:
# Optional — only run if you want to push to HuggingFace Hub

HF_REPO = 'YOUR_HF_USERNAME/the-pivot-qwen2.5-3b'  # change this

# from huggingface_hub import login
# login(token='hf_YOUR_TOKEN_HERE')
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f'Model pushed to: https://huggingface.co/{HF_REPO}')

print('Uncomment the lines above and set your HF_REPO to push the model.')

---
## Summary

| What | Value |
|------|-------|
| Model | Qwen2.5-3B-Instruct + LoRA |
| Training method | GRPO (Group Relative Policy Optimization) |
| Curriculum | 5 tiers: easy → very hard scenarios |
| W&B project | models-nexica-ai |
| Model saved | Google Drive → the_pivot_model/final_model |
| Estimated time | 90–120 min on Colab T4 |

After training, go to **W&B dashboard** to see:
- Reward curve (should trend upward)
- Curriculum tier progression
- Survival rate improvement
- Pivot timing score
